# 🚀 JalanCerdas AI — Retrain Model

### Cara Pakai:
1. **Runtime → Change runtime type → T4 GPU**
2. **Run semua cell** (Shift+Enter)
3. **Tunggu 30-50 menit**
4. **Download** model yang dihasilkan

⚠️ **JANGAN tutup tab selama training!**

## 1️⃣ Setup

In [ ]:
#@title Install & Check GPU
!pip install -q ultralytics kagglehub pyyaml

import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU: {name} ({mem:.1f} GB) ✅')
else:
    print('❌ NO GPU! Runtime → Change runtime type → T4 GPU')

## 2️⃣ Download & Prepare Dataset

In [ ]:
#@title Download + Prepare + Fix Path (satu cell)
import kagglehub, os, shutil, yaml
import xml.etree.ElementTree as ET
from pathlib import Path

# --- Download ---
print('1. Downloading dataset...')
src_path = Path(kagglehub.dataset_download('habibiahmadim09/kerusakan-jalan'))
print(f'   Source: {src_path}')

# Show structure
for split in ['train', 'valid', 'test']:
    d = src_path / split
    if d.exists():
        n = len(list(d.iterdir()))
        print(f'   {split}/: {n} items')

# --- Clone repo ---
if not Path('jalancerdas-ai').exists():
    !git clone -q https://github.com/Srjeff27/jalancerdas-ai.git

# --- Prepare ---
print('\n2. Preparing dataset...')
CLASS_MAP = {
    'retak_memanjang': 0,
    'pengelupasan_lapisan_permukaan': 1,
    'lubang': 2,
    'retak_kulit_buaya': 3,
    'retak_blok': 4,
    'retak_pinggir': 5,
}

dataset_dir = Path('jalancerdas-ai/ai-model/dataset')
if dataset_dir.exists():
    shutil.rmtree(dataset_dir)

total_images = 0
total_labels = 0

for src_split, dst_split in [('train', 'train'), ('valid', 'val'), ('test', 'test')]:
    src_dir = src_path / src_split
    if not src_dir.exists():
        print(f'   {src_split}/ not found, skipping')
        continue

    # Create output dirs
    img_dir = dataset_dir / dst_split / 'images'
    lbl_dir = dataset_dir / dst_split / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    # Find ALL images (recursive)
    images = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
        images.extend(src_dir.rglob(ext))

    # Find ALL XML annotations
    xml_map = {x.stem: x for x in src_dir.rglob('*.xml')}

    n_img = 0
    n_lbl = 0

    for img_path in images:
        # Copy image
        shutil.copy2(img_path, img_dir / img_path.name)
        n_img += 1

        # Find matching XML and convert
        xml_path = xml_map.get(img_path.stem)
        if xml_path and xml_path.exists():
            try:
                tree = ET.parse(xml_path)
                root = tree.getroot()
                w = int(root.find('size/width').text)
                h = int(root.find('size/height').text)

                lines = []
                for obj in root.findall('object'):
                    name = obj.find('name').text
                    if name not in CLASS_MAP:
                        continue
                    cls_id = CLASS_MAP[name]
                    bbox = obj.find('bndbox')
                    xmin = float(bbox.find('xmin').text)
                    ymin = float(bbox.find('ymin').text)
                    xmax = float(bbox.find('xmax').text)
                    ymax = float(bbox.find('ymax').text)
                    cx = ((xmin + xmax) / 2) / w
                    cy = ((ymin + ymax) / 2) / h
                    bw = (xmax - xmin) / w
                    bh = (ymax - ymin) / h
                    lines.append(f'{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')

                lbl_path = lbl_dir / (img_path.stem + '.txt')
                lbl_path.write_text('\n'.join(lines))
                n_lbl += len(lines)
            except Exception as e:
                print(f'   XML error: {xml_path.name}: {e}')
        else:
            # No annotation - write empty label
            (lbl_dir / (img_path.stem + '.txt')).write_text('')

    total_images += n_img
    total_labels += n_lbl
    print(f'   {dst_split}: {n_img} images, {n_lbl} labels')

print(f'\n   Total: {total_images} images, {total_labels} labels')

# --- Verify ---
print('\n3. Verifying...')
for s in ['train', 'val', 'test']:
    p = dataset_dir / s / 'images'
    if p.exists():
        n = len(list(p.glob('*')))
        status = '✅' if n > 0 else '❌'
        print(f'   {s}/images: {n} files {status}')
    else:
        print(f'   {s}/images: ❌ NOT FOUND')

# --- Fix data.yaml path ---
yaml_path = Path('jalancerdas-ai/ai-model/configs/data.yaml')
abs_path = str(Path.cwd() / 'jalancerdas-ai/ai-model/dataset')

with open(yaml_path) as f:
    data = yaml.safe_load(f)

data['path'] = abs_path

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print(f'\n   data.yaml path: {abs_path}')
print(f'   classes: {data["nc"]}')
print(f'   names: {list(data["names"].values())}')

# Save for next cells
DATA_YAML = str(yaml_path)
print('\n✅ READY TO TRAIN!')

## 3️⃣ Training

In [ ]:
#@title Train YOLOv8s (200 epochs, ~30-50 min)
from ultralytics import YOLO
import time

print('Training YOLOv8s...')
print('Estimated: 30-50 minutes')
print('DO NOT close this tab!\n')

start = time.time()

model = YOLO('yolov8s.pt')

results = model.train(
    data=DATA_YAML,
    epochs=200,
    batch=16,
    imgsz=640,
    device=0,
    patience=50,
    optimizer='SGD',
    lr0=0.01,
    lrf=0.01,
    mosaic=1.0,
    mixup=0.15,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    copy_paste=0.1,
    close_mosaic=15,
    project='jalancerdas-ai/ai-model/runs',
    name='train_v2',
    exist_ok=True,
)

elapsed = int(time.time() - start)
mins, secs = divmod(elapsed, 60)
hours, mins = divmod(mins, 60)

BEST_PT = 'jalancerdas-ai/ai-model/runs/train_v2/weights/best.pt'

print(f'\n{"="*50}')
print(f'  TRAINING COMPLETE in {hours}h {mins}m {secs}s')
print(f'  Best model: {BEST_PT}')
print(f'{"="*50}')

## 4️⃣ Evaluate

In [ ]:
#@title Evaluate Model
from ultralytics import YOLO

model = YOLO(BEST_PT)
results = model.val(data=DATA_YAML, device=0)

print('\n' + '='*50)
print('  EVALUATION RESULTS')
print('='*50)
print(f'  mAP@50:     {results.box.map50:.4f}')
print(f'  mAP@50-95:  {results.box.map:.4f}')
print(f'  Precision:  {results.box.mp:.4f}')
print(f'  Recall:     {results.box.mr:.4f}')

f1 = 2 * (results.box.mp * results.box.mr) / (results.box.mp + results.box.mr + 1e-8)
print(f'  F1 Score:   {f1:.4f}')

print(f'\n  Per-class mAP@50:')
for i, ap in enumerate(results.box.ap50):
    print(f'    {results.names[i]:<40} {ap:.4f}')
print('='*50)

## 5️⃣ Export & Download

In [ ]:
#@title Export TFLite + Download
import os
from google.colab import files

model = YOLO(BEST_PT)
export_path = model.export(
    format='tflite',
    imgsz=640,
    half=True,
    simplify=True,
)

size_mb = os.path.getsize(export_path) / (1024*1024)
print(f'TFLite: {export_path} ({size_mb:.1f} MB)')

print('\nDownloading files...')
files.download(export_path)
files.download(BEST_PT)

print('\n' + '='*50)
print('  ALL DONE!')
print('='*50)
print('\nNext steps:')
print('  1. Copy best.tflite → mobile/assets/models/pothole_yolo.tflite')
print('  2. flutter build apk --release')
print('  3. Test on phone!')